# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Classification.**

My lane (Ranking Signal Analysis) is fundamentally about *which signals matter*, not about
producing a final priority list -- so it doesn't map cleanly onto ranking/scoring (no single
score to output yet) or clustering (I'm not looking for page archetypes). To make "which signals
matter" concrete and testable, I'm framing it as a diagnostic classification problem: can each
candidate signal, on its own, separate pages that are currently declining from pages that
aren't, better than chance? That turns "signal analysis" from a vague EDA exercise into
something with an actual target, an actual metric, and a pass/fail bar per signal -- which is
what lets we compare signals honestly instead of eyeballing charts.


In [1]:
# Framing only -- no computation for this section.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy):** `is_declining_label = (trend_direction == "down")`, the same starter label
used in notebooks 01/02.

This is explicitly a **proxy, not a clean observed outcome** -- `trend_direction` is itself a
bucket computed from `trend_pct` over the current 90-day window, not a future-window result.
I'm using it here anyway because Section 1's goal is diagnostic (do signals separate the classes
at all?), not a final capstone claim. I'm flagging now, honestly, that a stronger version of this
target for later weeks would be a **future-window** label built from the warehouse's daily
table (e.g. features from a prior 90 days -> decline over the *next* 30 days), which the lane
guide calls out as the stronger alternative to this proxy. `trend_direction` and `trend_pct`
themselves are never used as features below -- only as the label, per the label-trap warning in
the data dictionary.


In [2]:
# Framing only -- no computation for this section.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: single-feature ROC-AUC against the proxy label**, computed direction-agnostically
(`max(auc, 1 - auc)`) since I don't want to assume in advance whether a signal moves up or down
with decline.

- 0.50 = the signal separates decliners from non-decliners no better than a coin flip.
- Meaningfully above 0.50 (I'll use >0.55 as my working bar, echoing the "which signals are
  worth building on" decision from ML-02) = the signal carries real, if modest, individual
  information about decline.

I picked single-feature AUC over a plain correlation because several of my candidate signals
(content_type, freshness_tier) are categorical/tiered, not purely linear, and AUC handles
monotonic-but-not-linear relationships better than Pearson correlation while still being
readable as "how separable are the two classes." It's also the same metric family
(`ROC-AUC, precision/recall vs base rate`) the framing skill lists for classification tasks, so
Section 5's model results stay comparable to this diagnostic pass.


In [3]:
# Framing only -- no computation for this section.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (already deduplicated by `content_id`), filtered the same way
the starter pipeline does (`impressions_90d > 0` and `content_age_days >= 90`), with the proxy
label attached as its own column. Loaded and shown below.


In [4]:
import os
import pandas as pd
import numpy as np

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same filter/dedupe the starter feature-prep step uses (see docs/lane guide, section 5)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"{len(df)} rows (= {len(df)} content pages), {df['client_id'].nunique()} clients")
print(f"Base rate of is_declining_label: {df['is_declining_label'].mean():.3f}\n")

cols_to_show = ["content_id", "client_id", "avg_position", "ctr", "days_since_last_update",
                 "engagement_rate", "word_count", "impressions_90d", "search_volume",
                 "trend_direction", "is_declining_label"]
df[cols_to_show].head(5)


30000 rows (= 30000 content pages), 32 clients
Base rate of is_declining_label: 0.542



,content_id,client_id,avg_position,ctr,days_since_last_update,engagement_rate,word_count,impressions_90d,search_volume,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.6,0.76,20,5.88,3221.0,3803,10.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,20.3,0.05,25,0.00,2481.0,15320,90.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,36.5,0.09,20,0.00,3515.0,12581,0.0,down,1
3,content_331d6c4de07b,client_19581e27de,6.2,0.49,22,1.28,NaN,11751,10.0,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,0.13,14,0.00,2803.0,19140,0.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The code below tests each candidate signal **on its own** against the proxy label. The result is
the actual argument for this lane: **every single signal, alone, is weak** -- direction-agnostic
AUC ranges from about 0.50 (engagement_rate, barely above chance) to about 0.58
(impressions_90d, the strongest single signal I tested). None of them, alone, would make a
defensible if-statement rule ("flag pages with X above threshold Y") -- the separation just isn't
there in any one column.

But notebook 02 already showed that **combining** several of these same weak signals into a
depth-2 decision tree lifted Precision@50 from 0.240 (hand rule) to 0.540, and a random forest
using all of them reached 0.740. That gap between "each signal alone: ~0.50-0.58 AUC" and
"combined: 0.74 precision@50" *is* the case for ML over a fixed rule here: no single column
carries enough signal to hand-write a rule around, but a model that can weigh several weak,
correlated signals together clearly captures something real that no single if-statement can.
A fixed rule would have to either pick one weak signal and underperform, or an engineer would
have to hand-tune multi-way interaction thresholds by trial and error -- which is exactly the
job a model does automatically and can be validated properly (client-holdout, as the starter
pipeline already does).


In [5]:
from sklearn.metrics import roc_auc_score

candidates = ["avg_position", "ctr", "days_since_last_update", "engagement_rate",
              "word_count", "impressions_90d", "search_volume"]

y = df["is_declining_label"]
results = []
for col in candidates:
    x = df[col].replace([np.inf, -np.inf], np.nan).fillna(df[col].median())
    auc = roc_auc_score(y, x)
    auc = max(auc, 1 - auc)  # direction-agnostic: how separable, regardless of sign
    results.append((col, round(auc, 3)))

auc_table = pd.DataFrame(results, columns=["signal", "single_feature_auc"]).sort_values(
    "single_feature_auc", ascending=False
)
print("Single-feature AUC vs is_declining_label (direction-agnostic; 0.50 = chance):")
print(auc_table.to_string(index=False))
print()
print("For comparison, from notebook 02 (already run, same starter data):")
print("  Hand rule Precision@50:      0.240")
print("  Depth-2 tree Precision@50:   0.540")
print("  Random forest Precision@50:  0.740  (per outputs/model_results.json)")


Single-feature AUC vs is_declining_label (direction-agnostic; 0.50 = chance):
                signal  single_feature_auc
       impressions_90d               0.584
         search_volume               0.560
            word_count               0.534
          avg_position               0.530
days_since_last_update               0.527
                   ctr               0.515
       engagement_rate               0.504

For comparison, from notebook 02 (already run, same starter data):
  Hand rule Precision@50:      0.240
  Depth-2 tree Precision@50:   0.540
  Random forest Precision@50:  0.740  (per outputs/model_results.json)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.